In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import save_image
import os

In [2]:
# 数据变换和加载
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
# MNIST 数据集下载
dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

In [3]:

class ConditionalGenerator(nn.Module):
    def __init__(self, num_classes):
        super(ConditionalGenerator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, 10)  # 假设标签嵌入的维度是10
        self.net = nn.Sequential(
            nn.Linear(100 + 10, 256),  # 100是噪声的维度，10是标签嵌入的维度
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 28*28),
            nn.Tanh()
        )
    
    def forward(self, z, labels):
        c = self.label_emb(labels)  # 嵌入标签
        z = torch.cat((z, c), dim=1)  # 将噪声和标签拼接
        return self.net(z).view(-1, 1, 28, 28)
class ConditionalDiscriminator(nn.Module):
    def __init__(self, num_classes):
        super(ConditionalDiscriminator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, 28*28)  # 标签嵌入的维度是28*28
        self.net = nn.Sequential(
            nn.Linear(28*28 + 28*28, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    
    def forward(self, img, labels):
        img_flat = img.view(img.size(0), -1)  # 展平图像
        c = self.label_emb(labels)  # 嵌入标签
        c = c.view(c.size(0), -1)  # 使标签嵌入与图像展平后的维度匹配
        input = torch.cat((img_flat, c), dim=1)  # 拼接图像和标签
        return self.net(input)

In [4]:
# 定义标签数量（数字类别数量）
num_classes = 10

# 创建模型实例
generator = ConditionalGenerator(num_classes)
discriminator = ConditionalDiscriminator(num_classes)

# 检查是否有可用的 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 将模型移动到 GPU
generator.to(device)
discriminator.to(device)

# 损失函数和优化器
criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

# 创建保存图像的目录
save_dir = 'images'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 训练循环
num_epochs = 10
for epoch in range(num_epochs):
    for i, (imgs, labels) in enumerate(dataloader):
        batch_size = imgs.size(0)
        real_imgs = imgs.to(device)
        labels = labels.to(device)
        valid = torch.ones(batch_size, 1, device=device)
        fake = torch.zeros(batch_size, 1, device=device)
        
        # 训练生成器
        optimizer_G.zero_grad()
        z = torch.randn(batch_size, 100, device=device)
        gen_labels = torch.randint(0, num_classes, (batch_size,), device=device)
        gen_imgs = generator(z, gen_labels)
        g_loss = criterion(discriminator(gen_imgs, gen_labels), valid)
        g_loss.backward()
        optimizer_G.step()
        
        # 训练判别器
        optimizer_D.zero_grad()
        real_loss = criterion(discriminator(real_imgs, labels), valid)
        fake_loss = criterion(discriminator(gen_imgs.detach(), gen_labels), fake)
        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()
    
    print(f'Epoch [{epoch+1}/{num_epochs}] | D Loss: {d_loss.item()} | G Loss: {g_loss.item()}')

    # 保存生成的图像
    if (epoch + 1) % 10 == 0:
        save_image(gen_imgs.data[:25], os.path.join(save_dir, f'{epoch+1}.png'), nrow=5, normalize=True)


Epoch [1/10] | D Loss: 0.4281291961669922 | G Loss: 1.6039626598358154
Epoch [2/10] | D Loss: 0.3163297176361084 | G Loss: 1.685732364654541
Epoch [3/10] | D Loss: 0.3451814353466034 | G Loss: 1.5891481637954712
Epoch [4/10] | D Loss: 0.4964899718761444 | G Loss: 1.4812321662902832
Epoch [5/10] | D Loss: 0.42814964056015015 | G Loss: 1.1621239185333252
Epoch [6/10] | D Loss: 0.6123921871185303 | G Loss: 1.269916296005249
Epoch [7/10] | D Loss: 0.5133241415023804 | G Loss: 1.3398241996765137
Epoch [8/10] | D Loss: 0.41207683086395264 | G Loss: 1.1821917295455933
Epoch [9/10] | D Loss: 0.5551191568374634 | G Loss: 1.307918667793274
Epoch [10/10] | D Loss: 0.6187655925750732 | G Loss: 0.8528977632522583


In [5]:

# 生成特定数字的图像
def generate_specific_digit(generator, digit, device):
    generator.eval()  # 设置模型为评估模式
    with torch.no_grad():
        z = torch.randn(1, 100, device=device)
        label = torch.tensor([digit], device=device)
        gen_img = generator(z, label)
    return gen_img

# 生成数字 9 的图像
digit_to_generate = 9
gen_img = generate_specific_digit(generator, digit_to_generate, device)
gen_img = gen_img.cpu()  # 将生成图像移动回 CPU
save_image(gen_img, 'digit_9.png', normalize=True)
